In [14]:
from langchain.agents import create_agent, AgentState
from langchain.tools import tool, ToolRuntime
from langchain.messages import HumanMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

from dotenv import load_dotenv

load_dotenv()

# state本质是一个字典，但是结构不能改变
# 必须以字典的形式操作state
class CustomState(AgentState):
  """Agent的任务状态"""
  count: int

@tool
def update_model_call_count(runtime: ToolRuntime):
  """更新模型的调用次数统计"""

  count = runtime.state.get("count", 0)
  update = {
    "count": count + 1,
    "messages": [
      ToolMessage(
        content="更新count成功",
        tool_call_id=runtime.tool_call_id
      )
    ]
  }
  return Command(update=update)

agent = create_agent(
  model="deepseek-flash",
  tools=[update_model_call_count],
  state_schema=CustomState, # 告诉agent使用自定义的state
  checkpointer=InMemorySaver(),
  system_prompt="你是一个热心肠的助手，你需要调用工具记录模型调用次数"
)


In [17]:
config = {
  "configurable": { "thread_id": "thread_1" }
}
response = agent.invoke(
  {"messages": [HumanMessage("你好")]},
  config
)

for message in response["messages"]:
  message.pretty_print()

================================ Human Message =================================

你好
================================== Ai Message ==================================
Tool Calls:
  update_model_call_count (call_00_q6WUZAPFqzTLALcTPfJF4907)
 Call ID: call_00_q6WUZAPFqzTLALcTPfJF4907
  Args:
================================= Tool Message =================================
Name: update_model_call_count

更新count成功
================================== Ai Message ==================================

你好！很高兴见到你 😊 有什么我可以帮你的吗？无论是聊天、解答问题，还是其他需要，都可以跟我说。
================================ Human Message =================================

你好
================================== Ai Message ==================================
Tool Calls:
  update_model_call_count (call_00_KOmszGkLMUqiQBfWHtMv0406)
 Call ID: call_00_KOmszGkLMUqiQBfWHtMv0406
  Args:
================================= Tool Message =================================
Name: update_model_call_count

更新count成功
================================== Ai Message

In [21]:
state = agent.get_state(config)
type(state)
print(state.values["count"])

2
